# 03 - Train the reranker + persist artifacts

We train the sklearn LogisticRegression reranker on 80% of the Q&A and persist:
* `bm25.pkl`       - rank_bm25 index
* `dense.pkl`      - TF-IDF/SVD encoder + matrix
* `reranker.pkl`   - sklearn LR
* `corpus_index.parquet` - paragraph table aligned with the indexes

This notebook deliberately uses no external API. The same pipeline is exposed in
`enterprise_rag.models.main` for CI / batch use.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from enterprise_rag.models import (
    build_indexes, evaluate_recall_at_k, make_training_pairs, train_reranker,
)

sns.set_theme(context="notebook", style="whitegrid")

In [ ]:
corpus = pd.read_parquet("../data/processed/policy_corpus.parquet").reset_index(drop=True)
qa = pd.read_parquet("../data/processed/policy_qa.parquet").reset_index(drop=True)
print(corpus.shape, qa.shape)

## 1. Build BM25 + dense indexes

In [ ]:
bm25, enc, dense_mat = build_indexes(corpus)
print("dense matrix shape:", dense_mat.shape)

## 2. Train / val split (80 / 20)

In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(len(qa))
cut = int(0.8 * len(qa))
train_qa = qa.iloc[idx[:cut]].reset_index(drop=True)
test_qa = qa.iloc[idx[cut:]].reset_index(drop=True)
print(len(train_qa), len(test_qa))

## 3. Build training pairs for the reranker

In [ ]:
X, y = make_training_pairs(train_qa, corpus, bm25, enc, dense_mat)
print("X shape:", X.shape, " pos rate:", float(y.mean()))

## 4. Fit the LR reranker

In [ ]:
reranker = train_reranker(X, y)
coefs = pd.Series(reranker.coef_[0], index=X.columns).sort_values()
coefs

## 5. Reranker coefficient bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
coefs.plot(kind="barh", color=["#3b82f6" if v >= 0 else "#ef4444" for v in coefs])
ax.set_title("Reranker LR coefficients")
plt.tight_layout()
plt.show()

## 6. Sweep the BM25 / dense blend weight alpha on the held-out set

In [ ]:
alphas = np.linspace(0.0, 1.0, 11)
rows = []
for a in alphas:
    m = evaluate_recall_at_k(test_qa, corpus, bm25, enc, dense_mat,
                              reranker=None, alpha=float(a))
    rows.append({"alpha": float(a), **m})
sweep = pd.DataFrame(rows)
sweep

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sweep["alpha"], sweep["recall@k"], marker="o", color="#3b82f6", label="Recall@5")
ax.plot(sweep["alpha"], sweep["mrr@10"], marker="o", color="#10b981", label="MRR@10")
ax.set_xlabel("alpha (1 = pure BM25, 0 = pure dense)")
ax.set_title("Hybrid blend sweep on held-out")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Final retrieval scoreboard

In [ ]:
bm25_only = evaluate_recall_at_k(test_qa, corpus, bm25, enc, dense_mat, reranker=None, alpha=1.0)
hybrid_only = evaluate_recall_at_k(test_qa, corpus, bm25, enc, dense_mat, reranker=None, alpha=0.5)
with_rerank = evaluate_recall_at_k(test_qa, corpus, bm25, enc, dense_mat, reranker=reranker, alpha=0.5)
scoreboard = pd.DataFrame([
    {"system": "BM25 only", **bm25_only},
    {"system": "Hybrid (alpha=0.5)", **hybrid_only},
    {"system": "Hybrid + LR rerank", **with_rerank},
])
scoreboard

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
scoreboard.set_index("system")[["recall@k", "mrr@10"]].plot(kind="bar", ax=ax,
                                                              color=["#3b82f6", "#10b981"])
ax.set_title("Retrieval scoreboard")
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 8. Save artifacts

In [ ]:
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(bm25, MODEL_DIR / "bm25.pkl")
joblib.dump({"encoder": enc, "matrix": dense_mat}, MODEL_DIR / "dense.pkl")
joblib.dump(reranker, MODEL_DIR / "reranker.pkl")
corpus.to_parquet(MODEL_DIR / "corpus_index.parquet", index=False)
print("saved -> ../models")